<a href="https://colab.research.google.com/github/jackc03/CORDIC_Simulation/blob/main/SCREEn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRNN-Assisted Video Upscaling ASIC  
### A Hardware/Software Codesign Walk-Through

Welcome! This notebook is the companion journal for my **hardware/software co-design project**: an **ASIC accelerator that upgrades 720 p video streams to 1080 p in real time** using a **Convolutional Recurrent Neural Network (CRNN)**.  
The goal is to show—step by step—how machine-learning research, algorithm engineering, RTL design, and physical-design constraints converge into a single silicon-ready pipeline.

---

## Motivation & Problem Statement
- **Bandwidth bottleneck:** Mobile and embedded devices often downlink only 720 p to save bandwidth or storage.  
- **Quality gap:** Naïve spatial upscalers (bilinear/nearest) yield soft edges and ringing artifacts.  
- **Opportunity:** A compact CRNN can *learn* spatio-temporal correlations to hallucinate sharper textures, delivering near-native 1080 p quality at a fraction of the bitrate.  
- **Challenge:** Deep models are compute-hungry. Achieving **⩾ 30 fps at 1080 p** within a **< 2 W power envelope** and **2 mm² core area** (SKY130 180 MHz budget) demands *co-optimized* hardware and software.

---

## Dataset
- **Dataset:** [DAVIS-2017 Unsupervised, Train/Val, Full-Resolution]—only raw RGB frames.  
  *LR frames* are generated on the fly via bicubic ↓ in the `Dataset` class.  


## 2 Dataset Generation & Processing

This section sets up everything we need to feed the **CRNN upscaler** with clean, memory-friendly training data.

### 2.1 Source Material – DAVIS-2017 (Unsupervised, Full-Resolution)
* • **60 train** + **30 val** video sequences, delivered as raw RGB frames  
* • Stored under `datasets/DAVIS_4K/⟨seq⟩/*.jpg`.

### 2.2 On-the-Fly LR/HR Pair Creation
1. **Bicubic downscale** to (480 p, 1440 p) to create the LR, HR counterparts.  
2. Assemble a **(prev, curr, next, hr) tuple** for temporal context.

> *Why dynamic downscaling instead of stored LR copies?*  
> Saves ~4 GB of disk, lets us experiment with different scale factors, and guarantees perfect alignment.

### 2.3 DataLoader Blueprint
| Split | # Sequences | # Triplets* | Purpose |
|-------|-------------|------------:|---------|
| **Train** | 60 | ≈ 25 k | Back-prop & augmentation |
| **Val**   | 30 | ≈ 12 k | PSNR / SSIM checkpoints |
| *(Test set loaded later for final metrics.)* |

\* Triplet count ≈ frames × (1 – 2/N) after dropping first & last frame per sequence.

### 2.4 Sanity Checks
* **Shape assert:** `(B, 3, H, W)` for each LR frame, `(B, 3, 2H, 2W)` for HR.  
* **Quick PSNR** between bicubic LR↑ and HR to catch corrupted images.  
* Visual spot-checks (overlay montage) stored in `/logs/sanity/`.

---

Run the next code cell to build the `VideoTripletDataset`, instantiate **train/val DataLoaders**, and print a mini-batch summary.


In [62]:
%cd /content/drive/MyDrive/screen/

# ─── DAVIS-2017 UNSUPERVISED Train+Val (Full-Res) ─────────────────────────
!mkdir -p datasets



/content/drive/MyDrive/screen


In [63]:
# # # ================================================================
# # # Download all 4 K 8‑bit YUV‑420 archives from UltraVideo Group
# # # Saves to  /content/screen/datasets/UVG/native_4k/
# # # ================================================================

# %cd /content/drive/MyDrive/screen/datasets
# !pip -q install tqdm py7zr

# import requests, os, sys, glob, py7zr
# from pathlib import Path
# from tqdm.auto import tqdm

# DEST_DIR = Path("UVG/zips_4k")
# DEST_DIR.mkdir(parents=True, exist_ok=True)
# EXTRACT_DIR = Path("UVG/native_4k")
# EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# # print("Download directory:", DEST_DIR.resolve())

# # # ---- explicit URL list (16 clips, 8‑bit 4:2:0) -----------------
# # URLS = [
# #     # 120 fps clips
# #     "https://ultravideo.fi/video/Beauty_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/Bosphorus_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/HoneyBee_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/Jockey_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/ReadySetGo_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/ShakeNDry_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/YachtRide_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/Lips_3840x2160_120fps_420_8bit_YUV_RAW.7z",
# #     # 50 fps clips
# #     "https://ultravideo.fi/video/CityAlley_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/FlowerFocus_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/FlowerKids_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/FlowerPan_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/RaceNight_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/RiverBank_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/SunBath_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# #     "https://ultravideo.fi/video/Twilight_3840x2160_50fps_420_8bit_YUV_RAW.7z",
# # ]

# # print(f"Downloading {len(URLS)} UVG clips.")

# # CHUNK = 1 << 23  # 8 MB

# # def download(url: str, dst: Path):
# #     with requests.get(url, stream=True) as r:
# #         r.raise_for_status()
# #         total = int(r.headers.get("content-length", 0))
# #         bar   = tqdm(total=total, unit="B", unit_scale=True, desc=dst.name)
# #         with dst.open("wb") as f:
# #             for blk in r.iter_content(CHUNK):
# #                 if blk:
# #                     f.write(blk)
# #                     bar.update(len(blk))
# #         bar.close()

# # # ---- fetch all clips ---------------------------------------------------------
# # for url in URLS:
# #     out_path = DEST_DIR / Path(url).name
# #     if out_path.exists():
# #         print(f"✔ {out_path.name} already present – skip")
# #         continue
# #     print(f"⬇ Downloading {out_path.name}")
# #     try:
# #         download(url, out_path)
# #     except Exception as e:
# #         print(f"✖ Error while downloading {out_path.name}: {e}")
# #         if out_path.exists():
# #             out_path.unlink()

# # print("\n✅  Finished. Archives saved in", DEST_DIR.resolve())

# for apath in glob.glob(os.path.join(DEST_DIR, "*.7z")):
#     sentinel = EXTRACT_DIR / (Path(apath).stem)
#     if sentinel.exists():
#         print(f"✔ {Path(apath).name} already extracted – skip")
#         continue

#     print(f"⇢ Extracting {Path(apath).name} …")
#     try:
#         with py7zr.SevenZipFile(apath, mode='r') as z:
#             z.extractall(path=EXTRACT_DIR)
#         # sentinel.touch()              # mark as done
#         # os.remove(apath)              # delete archive *now*
#     except Exception as e:
#         print(f"✖ Extraction failed on {apath}: {e}")
#         # keep the .7z so we can resume next run


# !rm -rf {DEST_DIR}
# %cd ..

In [64]:
# %cd /content/drive/MyDrive/screen/
# # ───────────────────────────────────────────────────────────────
# # Adjustable parameters 🖉
# INPUT_FOLDER  = "datasets/UVG/native_4k/"   # folder containing .yuv clips
# INPUT_RES     = "4k"                  # "4k"  (3840×2160) or "1080p" (1920×1080)
# OUTPUT_FOLDER = "datasets/UVG/downscaled_270p/"     # destination for 480×270 clips
# PIX_FMT       = "yuv420p"             # change if e.g. yuv420p10le
# # ───────────────────────────────────────────────────────────────

# from pathlib import Path
# import subprocess, shutil, sys

# PRESETS = {"4k": (3840, 2160), "1080p": (1920, 1080)}
# if INPUT_RES not in PRESETS:
#     raise ValueError("INPUT_RES must be '4k' or '1080p'")

# IN_W, IN_H   = PRESETS[INPUT_RES]
# OUT_W, OUT_H = 480, 270

# if not shutil.which("ffmpeg"):
#     sys.exit("❌ ffmpeg executable not found in PATH.")

# in_dir  = Path(INPUT_FOLDER).expanduser().resolve()
# out_dir = Path(OUTPUT_FOLDER).expanduser().resolve()
# out_dir.mkdir(exist_ok=True, parents=True)

# yuv_files = sorted(in_dir.glob("*.yuv"))
# if not yuv_files:
#     sys.exit(f"❌ No .yuv files in {in_dir}")

# print(f"Found {len(yuv_files)} files → output to {out_dir}")

# def downscale(src: Path, dst: Path):
#     cmd = [
#         "ffmpeg", "-y",
#         "-s", f"{IN_W}x{IN_H}", "-pix_fmt", PIX_FMT,
#         "-f", "rawvideo", "-i", str(src),
#         "-vf", f"scale={OUT_W}:{OUT_H}:flags=lanczos",
#         "-pix_fmt", PIX_FMT, "-f", "rawvideo", str(dst)
#     ]
#     subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

# for src in yuv_files:
#     dst = out_dir / src.name
#     if dst.exists():
#         print(f"· Skip {src.name} (already done)")
#         continue
#     print(f"· {src.name} → {dst.name}")
#     try:
#         downscale(src, dst)
#     except subprocess.CalledProcessError as e:
#         print(f"  ! ffmpeg error on {src.name}:\n{e.stderr.decode()}\n")

# print("\n✅ Finished down‑scaling all clips.")


In [65]:

# # ════════════════════════════════════════════════════════════════
# # Rename files by replacing "3840x2160" with "480x270"
# # Set FOLDER to the directory you want to process.
# # The operation is NON‑recursive (only that directory level).
# # ════════════════════════════════════════════════════════════════

# from pathlib import Path
# import os

# FOLDER = "datasets/UVG/downscaled_270p"
# OLD_TXT = "3840x2160"
# NEW_TXT = "480x270"

# folder = Path(FOLDER).expanduser().resolve()
# assert folder.is_dir(), f"{folder} is not a directory"

# n_total = n_renamed = 0
# for fp in folder.iterdir():
#     if not fp.is_file():
#         continue
#     n_total += 1
#     if OLD_TXT in fp.name:
#         new_name = fp.name.replace(OLD_TXT, NEW_TXT)
#         new_path = fp.with_name(new_name)
#         if new_path.exists():
#             print(f"⚠ skip {fp.name} → target {new_name} already exists")
#             continue
#         fp.rename(new_path)
#         print(f"✓ {fp.name}  →  {new_name}")
#         n_renamed += 1

# print(f"\nDone. Scanned {n_total} files, renamed {n_renamed}.")


In [66]:
# %cd /content/drive/MyDrive/screen/
# # ════════════════════════════════════════════════════════════════
# # USER SETTINGS
# N_CLIPS          = 5             # how many random clips to inspect
# FRAMES_PER_CLIP  = 2             # how many random frames per clip
# PATCH_SIZE       = 160           # square crop •••
# LR_SIZE          = (480, 270)    # down‑scaled resolution
# HR_SIZE          = (1920, 1080)  # native 4 K resolution
# PIX_FMT          = "yuv420p"     # UVG: YUV420 8‑bit
# # ════════════════════════════════════════════════════════════════

# from pathlib import Path
# import random, os, re
# import numpy as np
# import cv2, matplotlib.pyplot as plt
# from skimage.metrics import structural_similarity as ssim
# from skimage.metrics import peak_signal_noise_ratio as psnr

# # ───────────────────────────────── locate datasets ────────────────────────────
# root   = Path("datasets/UVG")
# lr_dir = root / "downscaled_270p"
# hr_dir = root / "downscaled_1080p"
# assert lr_dir.is_dir() and hr_dir.is_dir(), "Expected folder structure missing."

# # Capture everything *before* the resolution token; ignore any optional suffix
# rx_lr = re.compile(r"^(.*?)_480x270(?:_.*)?$")
# rx_hr = re.compile(r"^(.*?)_1920x1080(?:_.*)?$")

# lr_clips = {m.group(1): f
#             for f in lr_dir.glob("*.yuv") if (m := rx_lr.match(f.stem))}
# hr_clips = {m.group(1): f
#             for f in hr_dir.glob("*.yuv") if (m := rx_hr.match(f.stem))}

# common_keys = sorted(set(lr_clips) & set(hr_clips))
# assert common_keys, "No matching LR↔HR clip pairs found."

# random.seed(0)
# selected_keys   = random.sample(common_keys, min(N_CLIPS, len(common_keys)))
# selected_pairs  = [(lr_clips[k], hr_clips[k]) for k in selected_keys]

# W_lr, H_lr = LR_SIZE
# W_hr, H_hr = HR_SIZE
# bytes_lr   = W_lr * H_lr * 3 // 2              # 4:2:0 frame size
# bytes_hr   = W_hr * H_hr * 3 // 2

# def read_y(path: Path, w: int, h: int, idx: int) -> np.ndarray:
#     """Read the Y plane of frame `idx` from a planar YUV420 file."""
#     with path.open("rb") as f:
#         f.seek(idx * w * h * 3 // 2)
#         y = np.frombuffer(f.read(w * h), np.uint8).reshape(h, w)
#     return y.astype(np.float32) / 255.0

# def upscale(img, size, interp):
#     return cv2.resize(img, size, interpolation=interp)

# print(f"Selected {len(selected_pairs)} clips: {[p.name for p, _ in selected_pairs]}")

# # ─────────────────────────────── iterate over clips ───────────────────────────
# for lr_path, hr_path in selected_pairs:
#     clip = lr_path.name
#     total_frames = os.path.getsize(lr_path) // bytes_lr
#     frames = random.sample(range(total_frames),
#                            min(FRAMES_PER_CLIP, total_frames))
#     print(f"\n=== {clip}  •  frames: {frames}")

#     for f_idx in frames:
#         # ───── read Y planes ──────────────────────────────────────────────────
#         Y_lr = read_y(lr_path, W_lr, H_lr, f_idx)
#         Y_hr = read_y(hr_path, W_hr, H_hr, f_idx)

#         # ───── simple up‑scales ───────────────────────────────────────────────
#         Y_nn  = upscale(Y_lr, (W_hr, H_hr), cv2.INTER_NEAREST)
#         Y_bil = upscale(Y_lr, (W_hr, H_hr), cv2.INTER_LINEAR)

#         # ───── random patch coord ────────────────────────────────────────────
#         x0 = random.randint(0, W_hr - PATCH_SIZE)
#         y0 = random.randint(0, H_hr - PATCH_SIZE)
#         xs, ys = slice(x0, x0+PATCH_SIZE), slice(y0, y0+PATCH_SIZE)

#         patch_hr  = Y_hr [ys, xs]
#         patch_nn  = Y_nn [ys, xs]
#         patch_bil = Y_bil[ys, xs]

#         # ───── metrics ───────────────────────────────────────────────────────
#         psnr_nn  = psnr(patch_hr, patch_nn,  data_range=1.0)
#         ssim_nn  = ssim(patch_hr, patch_nn,  data_range=1.0)
#         psnr_bil = psnr(patch_hr, patch_bil, data_range=1.0)
#         ssim_bil = ssim(patch_hr, patch_bil, data_range=1.0)

#         diff_nn  = np.abs(patch_hr - patch_nn)
#         diff_bil = np.abs(patch_hr - patch_bil)

#         # ───── visualisation ────────────────────────────────────────────────
#         fig, axes = plt.subplots(2, 3, figsize=(10, 6))
#         axs = axes.ravel()

#         axs[0].imshow(patch_nn, cmap='gray')
#         axs[0].set_title(f"Nearest ↑\nPSNR {psnr_nn:.2f}  SSIM {ssim_nn:.4f}")
#         axs[1].imshow(patch_bil, cmap='gray')
#         axs[1].set_title(f"Bilinear ↑\nPSNR {psnr_bil:.2f}  SSIM {ssim_bil:.4f}")
#         axs[2].imshow(patch_hr, cmap='gray')
#         axs[2].set_title("Ground Truth")

#         im3 = axs[3].imshow(diff_nn, cmap='hot')
#         axs[3].set_title("|HR − Nearest↑|")
#         im4 = axs[4].imshow(diff_bil, cmap='hot')
#         axs[4].set_title("|HR − Bilinear↑|")
#         axs[5].axis("off")

#         for a in axs[:5]:
#             a.axis("off")
#         fig.colorbar(im4, ax=[axs[3], axs[4]], shrink=0.6, location='right')
#         fig.suptitle(f"{clip}  •  frame {f_idx}  •  patch ({x0},{y0})",
#                      fontsize=13)
#         plt.tight_layout()
#         plt.show()


In [67]:
%cd /content/drive/MyDrive/screen/

# ════════════════════════════════════════════════════════════════
# CONFIG
LR_SIZE = (480, 270)           # used only for YUV files
HR_SIZE = (1920, 1080)         # used only for YUV files
CLIP_DIR_LR = "datasets/UVG/downscaled_270p"      # can contain .yuv or images
CLIP_DIR_HR = "datasets/UVG/downscaled_1080p"     # can contain .yuv or images
SPLIT_RATIO = (0.70, 0.20, 0.10)
_RANDOM_SEED = 42
# ════════════════════════════════════════════════════════════════

from pathlib import Path
import re, os, cv2, numpy as np, torch, hashlib, math
from typing import List, Tuple, Optional, Dict
from torch.utils.data import Dataset
from collections import defaultdict

# ---- helpers ---------------------------------------------------
def read_yuv420(fp: Path, w: int, h: int, idx: int) -> np.ndarray:
    """Read one 8-bit YUV420 frame (I420/YU12) → float16 YUV, shape (3,H,W), range [0,1]."""
    frame_size = w * h * 3 // 2
    with fp.open('rb') as f:
        f.seek(idx * frame_size)
        y = np.frombuffer(f.read(w*h), np.uint8).reshape(h, w)
        u = np.frombuffer(f.read(w*h//4), np.uint8).reshape(h//2, w//2)
        v = np.frombuffer(f.read(w*h//4), np.uint8).reshape(h//2, w//2)
    u = cv2.resize(u, (w, h), interpolation=cv2.INTER_NEAREST)
    v = cv2.resize(v, (w, h), interpolation=cv2.INTER_NEAREST)
    yuv = np.stack([y, u, v]).astype(np.float16) / 255.0
    return yuv  # (3, H, W)

def read_image_as_yuv(fp: Path) -> np.ndarray:
    """Read an image → float16 YUV, shape (3,H,W), range [0,1]."""
    img = cv2.imread(str(fp), cv2.IMREAD_COLOR)  # BGR uint8
    if img is None:
        raise FileNotFoundError(fp)
    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    yuv = yuv.astype(np.float32) / 255.0
    yuv = np.transpose(yuv, (2, 0, 1)).astype(np.float16)
    return yuv

def frame_count(fp: Path, w: int, h: int) -> int:
    """Count frames in an 8-bit YUV420 file."""
    return fp.stat().st_size // (w * h * 3 // 2)

# ---- indexing (pairs LR↔HR by basename, supports .yuv & images) -----------
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
VIDEO_EXTS = {".yuv"}  # extend if you add NV12, etc.

rx_res_suffix = re.compile(r"^(.*?)(?:_\d+x\d+)$")  # strips trailing _WxH if present

def key_from_path(p: Path) -> str:
    stem = p.stem
    m = rx_res_suffix.match(stem)
    return m.group(1) if m else stem

def index_dir(dir_path: str) -> Dict[str, Path]:
    out = {}
    for f in Path(dir_path).iterdir():
        if not f.is_file():
            continue
        ext = f.suffix.lower()
        if ext in IMAGE_EXTS | VIDEO_EXTS:
            out[key_from_path(f)] = f
    return out

lr_map = index_dir(CLIP_DIR_LR)  # key -> LR Path
hr_map = index_dir(CLIP_DIR_HR)  # key -> HR Path
common_keys = sorted(set(lr_map) & set(hr_map))
print("matched keys:", len(common_keys))

# Precomputed maps (Path -> key), to avoid slow/buggy lookups later
PATH_TO_KEY_LR: Dict[Path, str] = {v: k for k, v in lr_map.items()}
PATH_TO_KEY_HR: Dict[Path, str] = {v: k for k, v in hr_map.items()}

# (Optional) build full sample catalog (frames+images) for info only
def build_samples() -> List[Tuple[Path, Path, Optional[int]]]:
    samples: List[Tuple[Path, Path, Optional[int]]] = []
    W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
    for k in common_keys:
        lp, hp = lr_map[k], hr_map[k]
        l_ext, h_ext = lp.suffix.lower(), hp.suffix.lower()
        if l_ext in VIDEO_EXTS and h_ext in VIDEO_EXTS:
            n_lr = frame_count(lp, W_lr, H_lr)
            n_hr = frame_count(hp, W_hr, H_hr)
            assert n_lr == n_hr, f"Frame mismatch for {k}: {n_lr} vs {n_hr}"
            for i in range(n_lr):
                samples.append((lp, hp, i))
        elif l_ext in IMAGE_EXTS and h_ext in IMAGE_EXTS:
            samples.append((lp, hp, None))
        else:
            raise ValueError(f"Mismatched types for {k}: {lp} vs {hp}")
    return samples

ALL_SAMPLES = build_samples()
print("total samples (frames+images):", len(ALL_SAMPLES))

# ---- splitting utilities --------------------------------------
def _stable_u01(s: str, seed: int) -> float:
    h = hashlib.blake2b(f"{seed}:{s}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(h, "big") / 2**64

def _apportion(total: int, ratios) -> Tuple[int, int, int]:
    """Largest-remainder method to split integer `total` by ratios → (train, val, test)."""
    rs = [float(r) for r in ratios]
    rsum = sum(rs) if sum(rs) > 0 else 1.0
    quotas = [total * r / rsum for r in rs]
    floors = [int(math.floor(q)) for q in quotas]
    rem = total - sum(floors)
    order = sorted(range(3), key=lambda i: (quotas[i] - floors[i]), reverse=True)
    for i in range(rem):
        floors[order[i]] += 1
    return tuple(floors)

# ---- per-YUV proportional split (small per-key differences allowed) ----
def make_balanced_splits_proportional(include_images: bool = True):
    """
    For each key:
      - If YUV↔YUV: apportion its N frames to (train,val,test) using SPLIT_RATIO.
        Within that key, select frames by a stable hashed order (no consecutive bias).
      - If image↔image and include_images=True: assign by stable hash threshold.
    Returns dict('train'|'val'|'test' -> List[(lp,hp,idx)])
    """
    W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
    splits = {"train": [], "val": [], "test": []}
    p_tr = SPLIT_RATIO[0] / sum(SPLIT_RATIO)
    p_va = SPLIT_RATIO[1] / sum(SPLIT_RATIO)

    for k in common_keys:
        lp, hp = lr_map[k], hr_map[k]
        l_ext, h_ext = lp.suffix.lower(), hp.suffix.lower()

        if l_ext in VIDEO_EXTS and h_ext in VIDEO_EXTS:
            N = frame_count(lp, W_lr, H_lr)
            assert N == frame_count(hp, W_hr, H_hr), f"Frame mismatch for {k}"

            # per-key integer counts close to ratios
            n_tr, n_va, n_te = _apportion(N, SPLIT_RATIO)

            # deterministic per-frame order within this file
            order = list(range(N))
            order.sort(key=lambda i: _stable_u01(f"{k}:{i}", _RANDOM_SEED))

            tr_idx = order[:n_tr]
            va_idx = order[n_tr:n_tr+n_va]
            te_idx = order[n_tr+n_va:n_tr+n_va+n_te]

            splits["train"].extend((lp, hp, i) for i in tr_idx)
            splits["val"]  .extend((lp, hp, i) for i in va_idx)
            splits["test"] .extend((lp, hp, i) for i in te_idx)

        elif include_images and (l_ext in IMAGE_EXTS and h_ext in IMAGE_EXTS):
            # assign the single image pair by stable hash threshold
            r = _stable_u01(f"{k}:image", _RANDOM_SEED)
            if r < p_tr:
                splits["train"].append((lp, hp, None))
            elif r < p_tr + p_va:
                splits["val"].append((lp, hp, None))
            else:
                splits["test"].append((lp, hp, None))
        else:
            raise ValueError(f"Mismatched types for {k}: {lp} vs {hp}")

    return splits

# ---- dataset ---------------------------------------------------
class LRHRDataset(Dataset):
    """Each item is one FRAME (from YUV) or one IMAGE pair: returns (lr, hr) as YUV tensors."""
    def __init__(self, samples: List[Tuple[Path, Path, Optional[int]]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        lp, hp, fi = self.samples[idx]
        if lp.suffix.lower() == ".yuv":
            W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
            lr = read_yuv420(lp, W_lr, H_lr, fi)
            hr = read_yuv420(hp, W_hr, H_hr, fi)
        else:
            lr = read_image_as_yuv(lp)
            hr = read_image_as_yuv(hp)
        lr = torch.from_numpy(lr)   # float16
        hr = torch.from_numpy(hr)   # float16
        return lr, hr

# ---- build proportional splits & datasets ----------------------
splits = make_balanced_splits_proportional(include_images=True)

train_ds = LRHRDataset(splits["train"])
val_ds   = LRHRDataset(splits["val"])
test_ds  = LRHRDataset(splits["test"])

print("train samples:", len(train_ds))
print("val   samples:", len(val_ds))
print("test  samples:", len(test_ds))

# ---- sanity: per-key ratios close to SPLIT_RATIO (uses precomputed map) ---
def per_key_counts(ds: LRHRDataset) -> Dict[str, int]:
    c = defaultdict(int)
    for lp, hp, i in ds.samples:
        c[PATH_TO_KEY_LR[lp]] += 1   # Path -> key via precomputed map
    return c

def per_key_shares():
    tr = per_key_counts(train_ds)
    va = per_key_counts(val_ds)
    te = per_key_counts(test_ds)
    keys = sorted(set(tr) | set(va) | set(te))
    if not keys:
        return
    def rng(a): return (min(a), max(a))
    tr_sh = [tr.get(k,0)/(tr.get(k,0)+va.get(k,0)+te.get(k,0)) for k in keys]
    va_sh = [va.get(k,0)/(tr.get(k,0)+va.get(k,0)+te.get(k,0)) for k in keys]
    te_sh = [te.get(k,0)/(tr.get(k,0)+va.get(k,0)+te.get(k,0)) for k in keys]
    print(f"per-key train share range: {rng(tr_sh)[0]:.3f}-{rng(tr_sh)[1]:.3f}")
    print(f"per-key val   share range: {rng(va_sh)[0]:.3f}-{rng(va_sh)[1]:.3f}")
    print(f"per-key test  share range: {rng(te_sh)[0]:.3f}-{rng(te_sh)[1]:.3f}")

per_key_shares()

# Optional: quick breakdown
n_train_frames = sum(1 for _,_,i in train_ds.samples if i is not None)
n_val_frames   = sum(1 for _,_,i in val_ds.samples if i is not None)
n_test_frames  = sum(1 for _,_,i in test_ds.samples if i is not None)
n_train_imgs   = len(train_ds) - n_train_frames
n_val_imgs     = len(val_ds) - n_val_frames
n_test_imgs    = len(test_ds) - n_test_frames
print(f"[frames] train/val/test: {n_train_frames}/{n_val_frames}/{n_test_frames}")
# print(f"[images] train/val/test: {n_train_imgs}/{n_val_imgs}/{n_test_imgs}")


/content/drive/MyDrive/screen
matched keys: 7
total samples (frames+images): 3900
train samples: 2730
val   samples: 780
test  samples: 390
per-key train share range: 0.700-0.700
per-key val   share range: 0.200-0.200
per-key test  share range: 0.100-0.100
[frames] train/val/test: 2730/780/390


In [68]:
%cd /content/drive/MyDrive/screen/

# ════════════════════════════════════════════════════════════════
# CONFIG
LR_SIZE = (480, 270)           # used only for YUV files
HR_SIZE = (1920, 1080)         # used only for YUV files
CLIP_DIR_LR = "datasets/UVG/downscaled_270p"      # can contain .yuv or images
CLIP_DIR_HR = "datasets/UVG/downscaled_1080p"     # can contain .yuv or images
SPLIT_RATIO = (0.70, 0.20, 0.10)
_RANDOM_SEED = 42
# ════════════════════════════════════════════════════════════════

from pathlib import Path
import re, os, cv2, numpy as np, torch, hashlib, math
from typing import List, Tuple, Optional, Dict
from torch.utils.data import Dataset

# ---- helpers ---------------------------------------------------
def read_yuv420(fp: Path, w: int, h: int, idx: int) -> np.ndarray:
    """Read one 8-bit YUV420 frame (I420/YU12) → float16 YUV, shape (3,H,W), range [0,1]."""
    frame_size = w * h * 3 // 2
    with fp.open('rb') as f:
        f.seek(idx * frame_size)
        y = np.frombuffer(f.read(w*h), np.uint8).reshape(h, w)
        u = np.frombuffer(f.read(w*h//4), np.uint8).reshape(h//2, w//2)
        v = np.frombuffer(f.read(w*h//4), np.uint8).reshape(h//2, w//2)
    u = cv2.resize(u, (w, h), interpolation=cv2.INTER_NEAREST)
    v = cv2.resize(v, (w, h), interpolation=cv2.INTER_NEAREST)
    yuv = np.stack([y, u, v]).astype(np.float16) / 255.0
    return yuv  # (3, H, W)

def read_image_as_yuv(fp: Path) -> np.ndarray:
    """Read an image → float16 YUV, shape (3,H,W), range [0,1]."""
    img = cv2.imread(str(fp), cv2.IMREAD_COLOR)  # BGR uint8
    if img is None:
        raise FileNotFoundError(fp)
    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    yuv = yuv.astype(np.float32) / 255.0
    yuv = np.transpose(yuv, (2, 0, 1)).astype(np.float16)
    return yuv

def frame_count(fp: Path, w: int, h: int) -> int:
    """Count frames in an 8-bit YUV420 file."""
    return fp.stat().st_size // (w * h * 3 // 2)

# ---- indexing (pairs LR↔HR by basename, supports .yuv & images) -----------
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
VIDEO_EXTS = {".yuv"}  # extend if you add NV12, etc.

rx_res_suffix = re.compile(r"^(.*?)(?:_\d+x\d+)$")  # strips trailing _WxH if present

def key_from_path(p: Path) -> str:
    stem = p.stem
    m = rx_res_suffix.match(stem)
    return m.group(1) if m else stem

def index_dir(dir_path: str) -> Dict[str, Path]:
    out = {}
    for f in Path(dir_path).iterdir():
        if not f.is_file():
            continue
        ext = f.suffix.lower()
        if ext in IMAGE_EXTS | VIDEO_EXTS:
            out[key_from_path(f)] = f
    return out

lr_map = index_dir(CLIP_DIR_LR)
hr_map = index_dir(CLIP_DIR_HR)
common_keys = sorted(set(lr_map) & set(hr_map))
print("matched keys:", len(common_keys))

# (Optional) build full sample catalog (frames+images) for info only
def build_samples() -> List[Tuple[Path, Path, Optional[int]]]:
    samples: List[Tuple[Path, Path, Optional[int]]] = []
    W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
    for k in common_keys:
        lp, hp = lr_map[k], hr_map[k]
        l_ext, h_ext = lp.suffix.lower(), hp.suffix.lower()
        if l_ext in VIDEO_EXTS and h_ext in VIDEO_EXTS:
            n_lr = frame_count(lp, W_lr, H_lr)
            n_hr = frame_count(hp, W_hr, H_hr)
            assert n_lr == n_hr, f"Frame mismatch for {k}: {n_lr} vs {n_hr}"
            for i in range(n_lr):
                samples.append((lp, hp, i))
        elif l_ext in IMAGE_EXTS and h_ext in IMAGE_EXTS:
            samples.append((lp, hp, None))
        else:
            raise ValueError(f"Mismatched types for {k}: {lp} vs {hp}")
    return samples

ALL_SAMPLES = build_samples()
print("total samples (frames):", len(ALL_SAMPLES))

# ---- splitting utilities --------------------------------------
def _stable_u01(s: str, seed: int) -> float:
    h = hashlib.blake2b(f"{seed}:{s}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(h, "big") / 2**64

def _apportion(total: int, ratios) -> Tuple[int, int, int]:
    """Largest-remainder method to split integer `total` by ratios → (train, val, test)."""
    rs = [float(r) for r in ratios]
    rsum = sum(rs) if sum(rs) > 0 else 1.0
    quotas = [total * r / rsum for r in rs]
    floors = [int(math.floor(q)) for q in quotas]
    rem = total - sum(floors)
    order = sorted(range(3), key=lambda i: (quotas[i] - floors[i]), reverse=True)
    for i in range(rem):
        floors[order[i]] += 1
    return tuple(floors)

# ---- per-YUV proportional split (small differences allowed) ----
def make_balanced_splits_proportional(include_images: bool = True):
    """
    For each key:
      - If YUV↔YUV: apportion its N frames to (train,val,test) using SPLIT_RATIO.
        Within that key, select frames by a stable hashed order (no consecutive bias).
      - If image↔image and include_images=True: assign by stable hash threshold.
    Returns dict('train'|'val'|'test' -> List[(lp,hp,idx)])
    """
    W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
    splits = {"train": [], "val": [], "test": []}

    for k in common_keys:
        lp, hp = lr_map[k], hr_map[k]
        l_ext, h_ext = lp.suffix.lower(), hp.suffix.lower()

        if l_ext in VIDEO_EXTS and h_ext in VIDEO_EXTS:
            N = frame_count(lp, W_lr, H_lr)
            assert N == frame_count(hp, W_hr, H_hr), f"Frame mismatch for {k}"

            # per-key integer counts close to ratios
            n_tr, n_va, n_te = _apportion(N, SPLIT_RATIO)

            # deterministic per-frame order within this file
            order = list(range(N))
            order.sort(key=lambda i: _stable_u01(f"{k}:{i}", _RANDOM_SEED))

            tr_idx = order[:n_tr]
            va_idx = order[n_tr:n_tr+n_va]
            te_idx = order[n_tr+n_va:n_tr+n_va+n_te]

            splits["train"].extend((lp, hp, i) for i in tr_idx)
            splits["val"]  .extend((lp, hp, i) for i in va_idx)
            splits["test"] .extend((lp, hp, i) for i in te_idx)

        elif include_images and (l_ext in IMAGE_EXTS and h_ext in IMAGE_EXTS):
            # assign the single image pair by stable hash threshold
            r = _stable_u01(f"{k}:image", _RANDOM_SEED)
            p_tr, p_va = SPLIT_RATIO[0]/sum(SPLIT_RATIO), SPLIT_RATIO[1]/sum(SPLIT_RATIO)
            if r < p_tr:
                splits["train"].append((lp, hp, None))
            elif r < p_tr + p_va:
                splits["val"].append((lp, hp, None))
            else:
                splits["test"].append((lp, hp, None))

        else:
            raise ValueError(f"Mismatched types for {k}: {lp} vs {hp}")

    return splits

# ---- dataset ---------------------------------------------------
class LRHRDataset(Dataset):
    """Each item is one FRAME (from YUV) or one IMAGE pair: returns (lr, hr) as YUV tensors."""
    def __init__(self, samples: List[Tuple[Path, Path, Optional[int]]]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        lp, hp, fi = self.samples[idx]
        if lp.suffix.lower() == ".yuv":
            W_lr, H_lr = LR_SIZE; W_hr, H_hr = HR_SIZE
            lr = read_yuv420(lp, W_lr, H_lr, fi)
            hr = read_yuv420(hp, W_hr, H_hr, fi)
        else:
            lr = read_image_as_yuv(lp)
            hr = read_image_as_yuv(hp)
        lr = torch.from_numpy(lr)   # float16
        hr = torch.from_numpy(hr)   # float16
        return lr, hr

# ---- build proportional splits & datasets ----------------------
splits = make_balanced_splits_proportional(include_images=True)

train_ds = LRHRDataset(splits["train"])
val_ds   = LRHRDataset(splits["val"])
test_ds  = LRHRDataset(splits["test"])

print("train samples:", len(train_ds))
print("val   samples:", len(val_ds))
print("test  samples:", len(test_ds))

# ---- sanity: per-key ratios are close to SPLIT_RATIO -----------
from collections import defaultdict
_inv_lr = {v: k for k, v in lr_map.items()}

def per_key_counts(ds: LRHRDataset):
    c = defaultdict(int)
    for lp, hp, i in ds.samples:
        c[_inv_lr[lp]] += 1
    return c

def per_key_shares():
    # compute share per key and show min/max across keys
    tr = per_key_counts(train_ds)
    va = per_key_counts(val_ds)
    te = per_key_counts(test_ds)
    shares = []
    for k in sorted(set(list(tr.keys()) + list(va.keys()) + list(te.keys()))):
        N = tr.get(k,0) + va.get(k,0) + te.get(k,0)
        if N == 0:
            continue
        shares.append((k, tr.get(k,0)/N, va.get(k,0)/N, te.get(k,0)/N))
    if shares:
        tr_min = min(s[1] for s in shares); tr_max = max(s[1] for s in shares)
        va_min = min(s[2] for s in shares); va_max = max(s[2] for s in shares)
        te_min = min(s[3] for s in shares); te_max = max(s[3] for s in shares)
        print(f"per-key train share range: {tr_min:.3f}-{tr_max:.3f}")
        print(f"per-key val   share range: {va_min:.3f}-{va_max:.3f}")
        print(f"per-key test  share range: {te_min:.3f}-{te_max:.3f}")

per_key_shares()

# Optional: quick breakdown
n_train_frames = sum(1 for _,_,i in train_ds.samples if i is not None)
n_val_frames   = sum(1 for _,_,i in val_ds.samples if i is not None)
n_test_frames  = sum(1 for _,_,i in test_ds.samples if i is not None)
n_train_imgs   = len(train_ds) - n_train_frames
n_val_imgs     = len(val_ds) - n_val_frames
n_test_imgs    = len(test_ds) - n_test_frames
print(f"[frames] train/val/test: {n_train_frames}/{n_val_frames}/{n_test_frames}")
# print(f"[images] train/val/test: {n_train_imgs}/{n_val_imgs}/{n_test_imgs}")


/content/drive/MyDrive/screen
matched keys: 7
total samples (frames): 3900
train samples: 2730
val   samples: 780
test  samples: 390
per-key train share range: 0.700-0.700
per-key val   share range: 0.200-0.200
per-key test  share range: 0.100-0.100
[frames] train/val/test: 2730/780/390


In [69]:
from skimage.metrics import structural_similarity as ssim_metric
import math
import torch
from typing import List, Tuple
import time
import numpy as np
from skimage.metrics import structural_similarity as ssim_metric
from skimage.color import rgb2lab, deltaE_ciede2000


import torch
import torch.nn.functional as F
import random




def random_patch_from_hr(hr: torch.Tensor, lr: torch.Tensor, scale: int = 4, patch_size: int = 256):
    """
    Crop a patch from HR first, then map to LR for alignment.

    Args:
        hr: (B, C, H_hr, W_hr) high-res tensor
        lr: (B, C, H_lr, W_lr) low-res tensor
        scale: upscaling factor (e.g., 4 for 270p→1080p)
        patch_size: crop size in HR pixels (default 256)

    Returns:
        lr_crop: cropped low-res tensor aligned to HR crop
        hr_crop: cropped high-res tensor
    """
    if isinstance(patch_size, (list, tuple, np.ndarray)):
        patch_size = int(patch_size[0])
    else:
        patch_size = int(patch_size)

    B, C, H_hr, W_hr = hr.shape
    ps_hr = patch_size
    ps_lr = patch_size // scale

    # Pick random top-left corner in HR space
    y_hr = random.randint(0, H_hr - ps_hr)
    x_hr = random.randint(0, W_hr - ps_hr)

    # Corresponding coordinates in LR space

    # TEST
    y_lr = min(y_hr // scale, max(0, H_hr - ps_lr))
    x_lr = min(x_hr // scale, max(0, lr.shape[2] - ps_lr))

    hr_crop = hr[:, :, y_hr:y_hr+ps_hr, x_hr:x_hr+ps_hr]
    lr_crop = lr[:, :, y_lr:y_lr+ps_lr, x_lr:x_lr+ps_lr]

    return lr_crop, hr_crop


# -----------------------------------------------------------------
# helper: tensor YUV → RGB  (expects values in 0-1)
# Y = 0-1 , U = 0-1 , V = 0-1
# -----------------------------------------------------------------
_YUV2RGB = torch.tensor([
    [1.0000,  0.0000,  1.4020   ],
    [1.0000, -0.344136, -0.714136],
    [1.0000,  1.7720,  0.0000   ],
], dtype=torch.float32)

def yuv_to_rgb(t: torch.Tensor) -> torch.Tensor:
    # move the matrix to the same device/dtype as input
    mat = _YUV2RGB.to(device=t.device, dtype=t.dtype)          # (3,3)
    y  = t[:, 0:1]
    u  = t[:, 1:2] - 0.5
    v  = t[:, 2:3] - 0.5
    yuv = torch.cat([y, u, v], dim=1)                          # (B,3,H,W)
    rgb = torch.tensordot(yuv, mat, dims=[[1],[1]])            # (B,H,W,3)
    rgb = rgb.permute(0, 3, 1, 2).clamp_(0, 1)                 # (B,3,H,W)
    return rgb


def to_bchw(arr):
    return torch.tensor(arr).permute(2, 0, 1).unsqueeze(0)  # (1,3,H,W)



# ---------- extra helpers -----------------------------------------------------

def _psnr_rgb(sr_rgb: np.ndarray, hr_rgb: np.ndarray) -> float:
    # inputs HxWx3 in [0,1]
    mse = np.mean((sr_rgb - hr_rgb) ** 2)
    return float("inf") if mse == 0 else 10.0 * np.log10(1.0 / mse)

def _ms_ssim_luma(sr_y: np.ndarray, hr_y: np.ndarray, scales=5) -> float:
    """
    Simple MS-SSIM (Wang'03) on luma; downsample by 2 with area every scale.
    Uses skimage's SSIM at each scale with Gaussian weighting.
    """
    # weights per Wang et al. (5 scales)
    weights = np.array([0.0448, 0.2856, 0.3001, 0.2363, 0.1333], dtype=np.float32)
    weights = weights[:scales]
    sr, hr = sr_y.copy(), hr_y.copy()
    vals = []
    for i in range(scales):
        ssim_i = ssim_metric(hr, sr, data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False, win_size=5)
        vals.append(ssim_i)
        # stop before last scale
        if i < scales - 1:
            # area downsample by 2
            H, W = sr.shape
            h2, w2 = H // 2, W // 2
            # reshape + average 2x2 blocks (area)
            sr = sr[:h2*2, :w2*2].reshape(h2, 2, w2, 2).mean(axis=(1,3))
            hr = hr[:h2*2, :w2*2].reshape(h2, 2, w2, 2).mean(axis=(1,3))
    # combine (product of SSIM^weight)
    vals = np.clip(np.array(vals, dtype=np.float64), 1e-6, 1.0)
    return float(np.prod(vals ** weights))

def _grad_psnr_luma(sr_y: np.ndarray, hr_y: np.ndarray) -> float:
    # Sobel gradients then PSNR on magnitude
    import cv2
    def sobel_mag(x):
        x8 = (x * 255.0).astype(np.uint8)
        gx = cv2.Sobel(x8, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(x8, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx*gx + gy*gy) / 255.0
        return mag
    ms = sobel_mag(sr_y); mh = sobel_mag(hr_y)
    mse = np.mean((ms - mh) ** 2)
    return float("inf") if mse == 0 else 10.0 * np.log10(1.0 / mse)

def _edge_psnr_luma(sr_y: np.ndarray, hr_y: np.ndarray) -> float:
    # PSNR only on Canny edge pixels of GT
    import cv2
    y8 = (hr_y * 255.0).astype(np.uint8)
    edges = cv2.Canny(y8, 80, 160) > 0
    if edges.sum() == 0:
        return float("nan")
    err = (sr_y - hr_y)[edges]
    mse = np.mean(err ** 2)
    return float("inf") if mse == 0 else 10.0 * np.log10(1.0 / mse)

def _deltae00(sr_rgb: np.ndarray, hr_rgb: np.ndarray) -> tuple[float, float]:
    # mean and 95th percentile ΔE00 (lower is better)
    lab_s = rgb2lab(np.clip(sr_rgb, 0, 1))
    lab_h = rgb2lab(np.clip(hr_rgb, 0, 1))
    de = deltaE_ciede2000(lab_h, lab_s)
    return float(np.mean(de)), float(np.percentile(de, 95))

def _try_lpips_setup(device_str="cpu"):
    try:
        import lpips
        net = lpips.LPIPS(net='alex').to(device_str).eval()
        for p in net.parameters():
            p.requires_grad_(False)
        return net
    except Exception:
        return None

def _lpips_score(lpips_net, sr_rgb: np.ndarray, hr_rgb: np.ndarray) -> float | None:
    if lpips_net is None:
        return None
    import torch
    # expects tensors in [-1,1], NCHW
    t = lambda a: torch.from_numpy(a).permute(2,0,1).unsqueeze(0).float()
    sr = t(np.clip(sr_rgb, 0, 1)) * 2 - 1
    hr = t(np.clip(hr_rgb, 0, 1)) * 2 - 1
    with torch.no_grad():
        val = lpips_net(sr, hr).item()
    return float(val)

def lanczos_up(t: torch.Tensor, sf=4) -> torch.Tensor:
    """
    t : (1, 3, H, W) torch tensor in [0,1] on any device/dtype
    return: (1, 3, H*sf, W*sf) on SAME device/dtype as t
    """
    t_np = t[0].detach().cpu().numpy()             # (3, H, W)
    C, H, W = t_np.shape
    # resize each channel with Lanczos-4 (OpenCV)
    up_ch = [cv2.resize(t_np[ch], (W*sf, H*sf), interpolation=cv2.INTER_LANCZOS4)
            for ch in range(C)]
    up_np = np.stack(up_ch, axis=0).astype(np.float32, copy=False)  # (3, H↑, W↑)
    up = torch.from_numpy(up_np).unsqueeze(0)        # (1, 3, H↑, W↑) on CPU
    return up.to(device=t.device, dtype=t.dtype)     # match caller

# ---------- modified demo -----------------------------------------------------
def demo(model, DataLoader, crp_sz=None, demo_samples=4):
    import cv2, time, numpy as np

    # normalize crp_sz to scalar int
    if isinstance(crp_sz, (list, tuple, np.ndarray)):
        crp_sz = int(crp_sz[0])
    elif crp_sz is not None:
        crp_sz = int(crp_sz)

    model.eval()
    demo_loader = DataLoader
    lpips_net = _try_lpips_setup(device_str="cpu")  # optional; None if not installed



    shown = 0
    for curr, hr in demo_loader:
        curr = curr.to(device).float()   # (1,3,Hlr,Wlr)
        hr   = hr.to(device).float()     # (1,3,Hhr,Whr)
        # infer scale
        Hlr, Wlr = curr.shape[-2:]
        Hhr, Whr = hr.shape[-2:]
        assert Hhr % Hlr == 0 and Whr % Wlr == 0, "Non-integer or non-uniform scale"
        scale = Hhr // Hlr

        # Choose LR display patch size:
        # - if crp_sz is given, treat it as HR size; adjust to be divisible by scale
        # - else sample LR size from {128:0.3, 96:0.5, 256:0.2} and set HR size = lr*scale
        if crp_sz is not None:
            patch_hr = (crp_sz // scale) * scale
            patch_lr = patch_hr // scale
        else:
            patch_lr = random.choices([128, 96, 256], weights=[0.3, 0.5, 0.2], k=1)[0]
            patch_hr = patch_lr * scale

        # pick aligned window (top-left on LR grid), clamp to bounds
        y_lr = random.randint(0, max(0, Hlr - patch_lr))
        x_lr = random.randint(0, max(0, Wlr - patch_lr))
        y_hr = y_lr * scale
        x_hr = x_lr * scale
        ys = slice(y_hr, y_hr + patch_hr)
        xs = slice(x_hr, x_hr + patch_hr)

        # timings
        if curr.is_cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                sr = model(curr)                          # (1,3,Hhr,Whr)
        if curr.is_cuda: torch.cuda.synchronize()
        t1 = time.perf_counter()

        # Bilinear & Lanczos baselines
        if curr.is_cuda: torch.cuda.synchronize()
        t2 = time.perf_counter()
        bil_up = F.interpolate(curr, scale_factor=scale, mode='bilinear', align_corners=False)  # (1,3,Hhr,Whr)
        if curr.is_cuda: torch.cuda.synchronize()
        t3 = time.perf_counter()

        t4 = time.perf_counter()
        lan_up = lanczos_up(curr, sf=scale)  # CPU tensor
        t5 = time.perf_counter()

        # RGB crops for display/metrics (HxWx3 in [0,1])
        crops = {
            "Bilinear ↑":   yuv_to_rgb(bil_up)[0, :, ys, xs].permute(1,2,0).cpu().numpy(),
            "Lanczos-4 ↑":  yuv_to_rgb(lan_up)[0, :, ys, xs].permute(1,2,0).cpu().numpy(),
            "Model SR":     yuv_to_rgb((sr + bil_up).clamp(0,1))[0, :, ys, xs].permute(1,2,0).cpu().numpy(),
            "Ground-truth": yuv_to_rgb(hr)[0, :, ys, xs].permute(1,2,0).cpu().numpy(),
        }
        ref = crops["Ground-truth"]

        # luma helpers
        Yw = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        toY = lambda rgb: np.clip((rgb * Yw).sum(-1), 0, 1)

        # metrics
        table = {}
        for k, img in crops.items():
            if k == "Ground-truth":
                continue
            y_pred, y_ref = toY(img), toY(ref)
            entry = {
                "PSNR_Y":       psnr(to_bchw(img), to_bchw(ref)),
                "PSNR_RGB":     _psnr_rgb(img, ref),
                "SSIM_Y":       ssim(to_bchw(img), to_bchw(ref)),
                "MS_SSIM_Y":    _ms_ssim_luma(y_pred, y_ref),
                "Grad_PSNR_Y":  _grad_psnr_luma(y_pred, y_ref),
                "Edge_PSNR_Y":  _edge_psnr_luma(y_pred, y_ref),
            }
            de_mean, de_p95 = _deltae00(img, ref)
            entry["ΔE00_mean"] = de_mean
            entry["ΔE00_P95"]  = de_p95

            lp = _lpips_score(lpips_net, img, ref)
            if lp is not None:
                entry["LPIPS"] = lp

            table[k] = entry

        # attach latencies (ms)
        table["Model SR"]["Latency_ms"]    = (t1 - t0) * 1000.0
        table["Bilinear ↑"]["Latency_ms"]  = (t3 - t2) * 1000.0
        table["Lanczos-4 ↑"]["Latency_ms"] = (t5 - t4) * 1000.0

        # ---- plot crops ----
        fig, ax = plt.subplots(1, 4, figsize=(14,4))
        for i, (k, v) in enumerate(crops.items()):
            ax[i].imshow(v)
            title = k
            if k != "Ground-truth":
                title += f"\nPSNR_Y {table[k]['PSNR_Y']:.2f}  SSIM_Y {table[k]['SSIM_Y']:.4f}"
            ax[i].set_title(title, fontsize=10)
            ax[i].axis("off")
        fig.suptitle(f"patch (x={x_hr}, y={y_hr})  {patch_hr}x{patch_hr}", fontsize=12, y=1.03)
        plt.tight_layout(); plt.show()

        # ---- print metric table ----
        headers = ["Method","Latency(ms)","PSNR_Y","PSNR_RGB","SSIM_Y","MS-SSIM_Y","Grad_PSNR_Y","Edge_PSNR_Y","ΔE00_mean","ΔE00_P95"]
        if lpips_net is not None:
            headers.append("LPIPS")
        rowfmt = "{:<12} {:>10.2f} {:>8.2f} {:>9.2f} {:>8.4f} {:>11.4f} {:>12.2f} {:>12.2f} {:>10.3f} {:>10.3f}"
        if lpips_net is not None:
            rowfmt += " {:>8.4f}"
        print("\n" + " | ".join(headers))
        for m in ["Bilinear ↑","Lanczos-4 ↑","Model SR"]:
            r = table[m]
            cells = [m, r["Latency_ms"], r["PSNR_Y"], r["PSNR_RGB"], r["SSIM_Y"], r["MS_SSIM_Y"],
                     r["Grad_PSNR_Y"], r["Edge_PSNR_Y"], r["ΔE00_mean"], r["ΔE00_P95"]]
            if lpips_net is not None:
                cells.append(r["LPIPS"])
            print(rowfmt.format(*cells))

        shown += 1
        if shown >= demo_samples:
            break


# ---------- 4. METRIC HELPERS -------------------------------------------------
_Y = torch.tensor([0.299,0.587,0.114]).view(1,3,1,1)
def _luma(t: torch.Tensor) -> torch.Tensor:
    return (t * _Y.to(t.device)).sum(1, keepdim=True)

def psnr(sr, hr):
    mse = F.mse_loss(_luma(sr), _luma(hr))
    return float("inf") if mse==0 else 10*math.log10(1.0/mse.item())

def ssim(sr, hr):
    a = _luma(sr).clamp(0,1).cpu().numpy()
    b = _luma(hr).clamp(0,1).cpu().numpy()
    return np.mean([ssim_metric(x.squeeze(), y.squeeze(), data_range=1.0)
                    for x,y in zip(a,b)])

@torch.no_grad()
def validate(model, loader, device, use_patches=True):
    model.eval()
    p_sum=s_sum=n=0
    with autocast(device_type='cuda', dtype=torch.bfloat16):
      for c,hr,_ in loader:

          if use_patches:
              c, hr = random_patch_from_hr(c,hr)

          c,hr = [t.to(device).type(torch.bfloat16) for t in (c,hr)]
          sr = model(c)[0]
          bs = p.size(0)
          p_sum += psnr(sr,hr)*bs
          s_sum += ssim(sr,hr)*bs
          n+=bs
    return p_sum/n, s_sum/n


def sobel_xy(x: torch.Tensor):
    # x: (B, C, H, W), dtype can be bf16/fp16/fp32
    C = x.shape[1]
    k = torch.tensor([[1, 0, -1],
                      [2, 0, -2],
                      [1, 0, -1]],
                     device=x.device, dtype=x.dtype)
    kx = k.view(1, 1, 3, 3).repeat(C, 1, 1, 1)          # (C,1,3,3)
    ky = k.t().contiguous().view(1, 1, 3, 3).repeat(C, 1, 1, 1)

    xpad = F.pad(x, (1, 1, 1, 1), mode='reflect')
    dx = F.conv2d(xpad, kx, groups=C)
    dy = F.conv2d(xpad, ky, groups=C)
    return dx, dy




def Yw_like(x: torch.Tensor):
    return x.new_tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)



def gradient_x(img):
    return img[:, :, :, :-1] - img[:, :, :, 1:]

def gradient_y(img):
    return img[:, :, :-1, :] - img[:, :, 1:, :]

def gradient_loss(sr, hr):
    loss_x = torch.mean(torch.abs(gradient_x(sr) - gradient_x(hr)))
    loss_y = torch.mean(torch.abs(gradient_y(sr) - gradient_y(hr)))
    return loss_x + loss_y



In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init

# ────────────────────────────────────────────────────────────────
# Building blocks
# ────────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """k×k residual block with projection skip; size-preserving."""
    def __init__(self, in_ch, out_ch: int, kernel=3, scale: float = 0.25, dilation: int = 1):
        super().__init__()
        self.scale = scale
        in_ch = int(in_ch)
        out_ch = int(out_ch)
        p = (dilation * (kernel - 1)) // 2  # keep spatial dims
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
        )
        # self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


def icnr(tensor: torch.Tensor, scale: int = 2, init_fn=init.kaiming_normal_):
    """ICNR initializer for weights feeding a PixelShuffle."""
    out_c, in_c, k, _ = tensor.shape
    if out_c % (scale ** 2) != 0:
        raise ValueError("ICNR init requires out_c divisible by scale²")
    subkernel = torch.zeros(out_c // (scale ** 2), in_c, k, k, device=tensor.device)
    init_fn(subkernel)
    subkernel = subkernel.repeat_interleave(scale ** 2, dim=0)
    tensor.data.copy_(subkernel)



class ConvBlock(nn.Module):
    """k×k residual-ish stack; size-preserving."""
    def __init__(self, in_ch, out_ch: int, kernel=3, scale: float = 0.25, dilation: int = 1):
        super().__init__()
        p = (dilation * (kernel - 1)) // 2
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, 1, p, dilation=dilation, bias=True),
        )
    def forward(self, x): return self.body(x)

class PSBlock(nn.Module):
    """Standard PixelShuffle upsampler: 3×3 conv → PixelShuffle(r) → act."""
    def __init__(self, in_ch, out_ch, scale=2, act=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * (scale**2), 3, 1, 1, bias=True)
        self.ps   = nn.PixelShuffle(scale)
        self.act  = nn.PReLU() if act else nn.Identity()
        if self.conv.bias is not None:
            nn.init.zeros_(self.conv.bias)
    def forward(self, x):
        x = self.conv(x)
        x = self.ps(x)
        return self.act(x)

def dw_sep(in_ch, out_ch, k=3):
    p = k // 2
    return nn.Sequential(
        nn.Conv2d(in_ch, in_ch, k, 1, p, groups=in_ch, bias=True),
        nn.PReLU(),
        nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=True),
        nn.PReLU(),
    )



class UpsampleBlock(nn.Module):
    """Depthwise-separable pre-PS: DW 3×3 + PW 1×1 → PixelShuffle(r). Size-preserving."""
    def __init__(self, in_ch, out_ch, scale=2, act=True):
        super().__init__()
        self.scale = scale
        self.dw = nn.Conv2d(in_ch, in_ch, 3, 1, 1, groups=in_ch, bias=True)                 # keeps H,W
        self.pw = nn.Conv2d(in_ch, out_ch * (scale**2), 1, 1, 0, bias=True)                 # keeps H,W
        icnr(self.pw.weight, scale=self.scale)
        if self.pw.bias is not None:
            nn.init.zeros_(self.pw.bias)
        self.act = nn.PReLU() if act else nn.Identity()

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = F.pixel_shuffle(x, self.scale)  # H,W → rH,rW ; channels → out_ch
        return self.act(x)


def dw_sep(in_ch, out_ch, k=3):
    """Depthwise-separable conv block, size-preserving."""
    p = k // 2
    return nn.Sequential(
        nn.Conv2d(in_ch, in_ch, k, 1, p, groups=in_ch, bias=True),  # keeps H,W
        nn.PReLU(),
        nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=True),               # keeps H,W
        nn.PReLU(),
    )



# ────────────────────────────────────────────────────────────────
# 4× VSR generator that outputs an HR residual (3ch) to add to bilinear upsample
# ────────────────────────────────────────────────────────────────
class Screen(nn.Module):
    """
    270p → 1080p (×4) generator that predicts an HR residual (B,3,4H,4W).
    Your training should do: sr = sr_res + F.interpolate(lr, scale_factor=4, ...).
    """

    def __init__(self, in_ch: int = 3, lr_ch: int = 256, hid_c: int = 128):
        super().__init__()
        self.in_ch = in_ch
        self.lr_ch = lr_ch

        # LR stem (size-preserving) — FIX: start from in_ch → lr_ch
        self.feat_in = nn.Sequential(
            nn.Conv2d(in_ch, lr_ch, 3, 1, 1, bias=True), nn.ReLU(inplace=True),
            nn.Conv2d(lr_ch, lr_ch, 3, 1, 1, bias=True),  nn.ReLU(inplace=True),
            nn.Conv2d(lr_ch, lr_ch, 3, 1, 1, bias=True), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, lr_ch, 5, 1, 2, bias=True) , nn.ReLU(inplace=True),
            nn.Conv2d(lr_ch, lr_ch, 3, 1, 1, bias=True),
        )

        # Skip from LR input → hid_c (to be added right before first upsample)
        self.feat_skip = nn.Sequential(
            nn.Conv2d(in_ch, hid_c, kernel_size=1, stride=1, padding=0, bias=True),
            nn.ReLU(inplace=True),
        )

        # LR residual refinement (size-preserving), grow width to hid_c
        self.res_lr = nn.Sequential(
            ConvBlock(lr_ch, lr_ch//4),
            ConvBlock(lr_ch//4, lr_ch//2),
            ConvBlock(lr_ch//2, lr_ch//2),
            ConvBlock(lr_ch//2, hid_c),   # ensure we land exactly at hid_c
            ConvBlock(hid_c, hid_c),
        )

        # Two ×2 PixelShuffle stages → total ×4
        self.up1 = UpsampleBlock(hid_c, 64, scale=2)   # (B,64,2H,2W)
        self.up2 = UpsampleBlock(64,   16, scale=2)    # FIX: in_ch should be 64, not hid_c

        # HR tail (size-preserving)
        self.hr_tail = nn.Sequential(
            dw_sep(16, 16, 3),
            dw_sep(16, 8,  3),
            dw_sep(8,  8,  3),
            dw_sep(8,  4,  3),
        )

        # Final 3×3 to 3-channel residual (zero-init)
        self.final = nn.Conv2d(4, 3, 3, 1, 1, bias=True)
        nn.init.zeros_(self.final.weight)
        nn.init.zeros_(self.final.bias)

    def forward(self, lr: torch.Tensor):
        """
        Returns:
            sr_res: (B,3,4H,4W) residual to be added to bilinear upsample outside.
        """
        x = self.feat_in(lr)        # (B,lr_ch,H,W)
        x = self.res_lr(x)          # (B,hid_c,H,W)

        feat_skip = self.feat_skip(lr)   # (B,hid_c,H,W)
        x = self.up1(x + feat_skip)      # (B,64,2H,2W)
        x = self.up2(x)                  # (B,32,4H,4W)
        x = self.hr_tail(x)              # (B,8,4H,4W)
        sr_res = self.final(x)           # (B,3,4H,4W)
        return sr_res


In [71]:
# ────────────────────────────────────────────────────────────────
# Imports
# ────────────────────────────────────────────────────────────────
import os
import time
from pathlib import Path, PurePosixPath
import random
from collections import deque
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
# cosine ease
def ease(x):
    return 0.5 - 0.5*math.cos(math.pi*x)


# ────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ────────────────────────────────────────────────────────────────
num_epochs      = 25
skip_warmup     = False
save_every_iter = 250

resume_G = None #"/content/drive/MyDrive/screen/trained_models/Adv_Perc_training_17Sep2025_0701/epoch_residual_avg.png"

save_root = os.path.join(Path("trained_models"), f"Adv_Perc_training_{time.strftime('%d%b%Y_%H%M')}")
os.makedirs(save_root, mode=0o777, exist_ok=True)

# ────────────────────────────────────────────────────────────────
# 2.  MODEL SETUP
# ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G = Screen().to(device)  # assumes Screen is defined elsewhere

def _pull_state(path: str | Path, key: str):
    obj = torch.load(path, map_location=device)
    if isinstance(obj, dict) and key in obj:
        return obj[key]
    return obj

if resume_G and Path(resume_G).is_file():
    G.load_state_dict(_pull_state(resume_G, "G"), strict=False)
    print("✓ loaded G from", PurePosixPath(resume_G).name)

# VGG19 perceptual features (up to conv4_4)
vgg = models.vgg19_bn(weights=models.VGG19_BN_Weights.IMAGENET1K_V1).features[:36].eval().to(device)
for p in vgg.parameters():
    p.requires_grad_(False)
mean, std = (torch.tensor(v, device=device).view(1, 3, 1, 1) for v in ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))

def _vgg_norm(x): return (x - mean) / std

# Optimiser & AMP
opt_G = torch.optim.Adam(G.parameters(), 2e-4, (0.9, 0.99))
scaler_G = GradScaler()

# Data loaders (assumes LRHRDataset & splits exist)
train_ds = LRHRDataset(splits["train"])
val_ds   = LRHRDataset(splits["val"])
train_ld = torch.utils.data.DataLoader(train_ds, 8, shuffle=True, num_workers=8, pin_memory=True)
val_ld   = torch.utils.data.DataLoader(val_ds, 1, shuffle=True, num_workers=8, pin_memory=True)

demo_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
PATCH_SET = (128, 196, 256)

# ────────────────────────────────────────────────────────────────
# 3. Plotter (residual drawn thicker but UNDERNEATH)
# ────────────────────────────────────────────────────────────────
class TrainPlotter:
    def __init__(self, out_dir, ema_beta=0.9, smooth_window=200):
        self.out = out_dir
        os.makedirs(self.out, exist_ok=True)

        # storage
        self.iters = []
        self.lossG = []
        self.perc  = []
        self.res   = []
        self.pix   = []
        self.grad  = []

        # weighted (exact terms used inside loss_G at that iter)
        self.perc_w = []
        self.res_w  = []
        self.pix_w  = []
        self.grad_w = []

        # running means
        self.ema_beta = ema_beta
        self.ema = None

        # ring buffer
        self.win = smooth_window
        self.buf_loss = deque(maxlen=self.win)

        # per-epoch aggregates
        self.epoch_idx  = []
        self.epoch_loss = []
        self.epoch_perc = []
        self.epoch_res  = []
        self.epoch_pix  = []
        self.epoch_grad = []

        self._last_epoch_end_idx = 0

    def update_iter(self, it, lossG, perc, res, pix, grad, perc_w, res_w, pix_w, grad_w):
        self.iters.append(it)
        self.lossG.append(lossG)
        self.perc.append(perc)
        self.res.append(res)
        self.pix.append(pix)
        self.grad.append(grad)

        self.perc_w.append(perc_w)
        self.res_w.append(res_w)
        self.pix_w.append(pix_w)
        self.grad_w.append(grad_w)

        # EMA on total loss
        self.ema = lossG if self.ema is None else (self.ema * self.ema_beta + lossG * (1 - self.ema_beta))
        self.buf_loss.append(lossG)

    def update_epoch(self, ep):
        import numpy as np
        self.epoch_idx.append(ep)
        start = getattr(self, "_last_epoch_end_idx", 0)
        end = len(self.lossG)

        if end > start:
            self.epoch_loss.append(float(np.mean(self.lossG[start:end])))
            self.epoch_perc.append(float(np.mean(self.perc[start:end])))
            self.epoch_res.append(float(np.mean(self.res[start:end])))
            self.epoch_pix.append(float(np.mean(self.pix[start:end])))
            self.epoch_grad.append(float(np.mean(self.grad[start:end])))
        else:
            exit(1)
            # duplicate previous if no new samples
            self.epoch_loss.append(self.epoch_loss[-1] if self.epoch_loss else 0.0)
            self.epoch_perc.append(self.epoch_perc[-1] if self.epoch_perc else 0.0)
            self.epoch_res.append(self.epoch_res[-1] if self.epoch_res else 0.0)
            self.epoch_pix.append(self.epoch_pix[-1] if self.epoch_pix else 0.0)
            self.epoch_grad.append(self.epoch_grad[-1] if self.epoch_grad else 0.0)

        self._last_epoch_end_idx = end

    # -------- plotting helpers --------
    def _save_plot(self, fname, x, series, labels, title, ylabel, ylim=None):
        import numpy as np
        X = np.asarray(x, dtype=float)

        # sanitize and pair
        items = []
        for y, lab in zip(series, labels):
            y = np.asarray(y, dtype=float)
            n = min(len(X), len(y))
            if n == 0:
                continue
            xv, yv = X[:n], y[:n]
            mask = np.isfinite(yv)
            if mask.any():
                items.append((xv[mask], yv[mask], lab))
        if not items:
            return

        # styles: residual is thicker but LOWER zorder and drawn first
        style = {
            "perc_raw":              {"color": "#1f77b4", "zorder": 3},
            "pixel_raw":             {"color": "#2ca02c", "zorder": 2},
            "grad_raw":              {"color": "#d62728", "zorder": 4},
            "residual_raw":          {"color": "#ff7f0e", "linewidth": 2.8, "zorder": 0, "alpha": 0.95},

            "perc_weighted":         {"color": "#1f77b4", "linestyle": "--", "zorder": 3},
            "pixel_weighted":        {"color": "#2ca02c", "linestyle": "--", "zorder": 2},
            "grad_weighted":         {"color": "#d62728", "linestyle": "--", "zorder": 4},
            "residual_weighted":     {"color": "#ff7f0e", "linestyle": "--", "linewidth": 2.8, "zorder": 0, "alpha": 0.95},
            "EMA(0.9)":              {"color": "#9467bd", "linestyle": ":", "zorder": 5},
        }

        # draw residual first (underneath)
        priority = {"residual_raw": 0, "residual_weighted": 0}  # others default to 1
        items.sort(key=lambda t: priority.get(t[2], 1))

        plt.figure()

        # robust y-limits
        if ylim is None:
            vals = np.concatenate([it[1] for it in items if len(it[1])])
            lo, hi = np.quantile(vals, [0.01, 0.99])
            span = max(1e-6, hi - lo)
            ylim = (max(0.0, lo - 0.1 * span), hi + 0.1 * span)

        for xv, yv, lab in items:
            plt.plot(xv, yv, label=lab, **style.get(lab, {}))

        plt.title(title)
        plt.xlabel("iteration")
        plt.ylabel(ylabel)
        plt.ylim(*ylim)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(self.out, fname), dpi=140)
        plt.close()

    def _ema_curve(self, series, beta=0.9):
        out, e = [], None
        for v in series:
            e = v if e is None else (beta * e + (1 - beta) * v)
            out.append(e)
        return out

    def _save_single_iter(self, series, name, ylabel, weighted=False):
        if not self.iters:
            return
        ema = self._ema_curve(series, beta=0.9)
        suffix = "weighted" if weighted else "raw"
        self._save_plot(
            f"iter_{name}_{suffix}.png",
            self.iters,
            [series, ema],
            [name, "EMA(0.9)"],
            f"{name} ({suffix}) vs Iteration",
            ylabel
        )

    # public plotters
    def save_individual_iter_plots(self):
        # raw
        self._save_single_iter(self.perc, "perc", "loss", weighted=False)
        self._save_single_iter(self.res,  "residual", "loss", weighted=False)
        self._save_single_iter(self.pix,  "pixel", "loss", weighted=False)
        self._save_single_iter(self.grad, "grad", "loss", weighted=False)
        # weighted
        self._save_single_iter(self.perc_w, "perc", "weighted loss", weighted=True)
        self._save_single_iter(self.res_w,  "residual", "weighted loss", weighted=True)
        self._save_single_iter(self.pix_w,  "pixel", "weighted loss", weighted=True)
        self._save_single_iter(self.grad_w, "grad", "weighted loss", weighted=True)

    def save_individual_epoch_plots(self):
        if not self.epoch_idx:
            return
        def _plot_epoch(y, name):
            plt.figure()
            plt.plot(self.epoch_idx, y, marker="o")
            plt.title(f"{name} per-epoch average")
            plt.xlabel("epoch")
            plt.ylabel(f"{name}_avg")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(self.out, f"epoch_{name}_avg.png"), dpi=140)
            plt.close()
        _plot_epoch(self.epoch_perc, "perc")
        _plot_epoch(self.epoch_res,  "residual")
        _plot_epoch(self.epoch_pix,  "pixel")
        _plot_epoch(self.epoch_grad, "grad")

    def save_iter_plots(self):
        ema_curve = self._ema_curve(self.lossG, beta=0.9)
        self._save_plot(
            "train_iter_loss.png",
            self.iters,
            [self.lossG, ema_curve],
            ["loss_G", "EMA(0.9)"],
            "Training Loss vs Iteration",
            "loss_G"
        )
        self._save_plot(
            "train_iter_components.png",
            self.iters,
            [self.perc, self.res, self.pix, self.grad],
            ["perc_raw", "residual_raw", "pixel_raw", "grad_raw"],
            "Raw Components vs Iteration",
            "loss"
        )
        self._save_plot(
            "train_iter_components_weighted.png",
            self.iters,
            [self.perc_w, self.res_w, self.grad_w],
            ["perc_weighted", "residual_weighted", "grad_weighted"], # Removed pixel_weighted as it's not in the list
            "Weighted Components vs Iteration",
            "weighted loss"
        )

    def save_iter_components_dualaxis(self):
        if not self.iters:
            return
        X = np.arange(len(self.iters))
        def _safe(y):
            y = np.asarray(y, dtype=float)
            n = min(len(X), len(y))
            return X[:n], y[:n]

        x1, perc = _safe(self.perc)
        x2, pix  = _safe(self.pix)
        x3, res  = _safe(self.res)
        x4, grd  = _safe(self.grad)

        plt.figure(figsize=(8,4))
        ax1 = plt.gca()
        l1, = ax1.plot(x1, perc, color="#1f77b4", label="perc_raw",  zorder=2)
        l2, = ax1.plot(x2, pix,  color="#2ca02c", label="pixel_raw", zorder=1)
        ax1.set_xlabel("iteration"); ax1.set_ylabel("perc/pixel"); ax1.grid(True, alpha=0.3)

        ax2 = ax1.twinx()
        l3, = ax2.plot(x3, res, color="#ff7f0e", label="residual_raw", linewidth=2.5, zorder=0)  # bottom
        l4, = ax2.plot(x4, grd, color="#d62728", label="grad_raw",      zorder=3)
        ax2.set_ylabel("residual/grad")

        lines = [l1, l2, l3, l4]; labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc="upper right")
        plt.title("Raw Components vs Iteration (dual y-axes)")
        plt.tight_layout()
        plt.savefig(os.path.join(self.out, "train_iter_components_dualaxis.png"), dpi=140)
        plt.close()

    def save_epoch_plots(self):
        if len(self.epoch_idx) == 0:
            return
        plt.figure()
        plt.plot(self.epoch_idx, self.epoch_loss, label="loss_G_avg")
        plt.plot(self.epoch_idx, self.epoch_perc, label="perc_avg")
        plt.plot(self.epoch_idx, self.epoch_res,  label="residual_avg")
        plt.plot(self.epoch_idx, self.epoch_pix,  label="pixel_avg")
        plt.plot(self.epoch_idx, self.epoch_grad, label="grad_avg")
        plt.title("Per-epoch Averages")
        plt.xlabel("epoch"); plt.ylabel("average loss")
        plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(self.out, "train_epoch_avg.png"), dpi=140)
        plt.close()

# ────────────────────────────────────────────────────────────────
# 4. Sobel loss (for grad term)
# ────────────────────────────────────────────────────────────────
def sobel_loss(sr_rgb: torch.Tensor,
               hr_rgb: torch.Tensor,
               mode: str = "mag",
               reduction: str = "l1"):
    Yw = sr_rgb.new_tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
    sr_y = (sr_rgb * Yw).sum(1, keepdim=True)
    hr_y = (hr_rgb * Yw).sum(1, keepdim=True)

    k = torch.tensor([[1, 0, -1],
                      [2, 0, -2],
                      [1, 0, -1]], device=sr_y.device, dtype=sr_y.dtype)
    kx = k.view(1,1,3,3); ky = k.t().contiguous().view(1,1,3,3)

    sr_y = F.pad(sr_y, (1,1,1,1), mode="reflect")
    hr_y = F.pad(hr_y, (1,1,1,1), mode="reflect")

    sr_dx = F.conv2d(sr_y, kx) / 8.0; sr_dy = F.conv2d(sr_y, ky) / 8.0
    hr_dx = F.conv2d(hr_y, kx) / 8.0; hr_dy = F.conv2d(hr_y, ky) / 8.0

    if mode == "xy":
        diff = (sr_dx - hr_dx).abs() + (sr_dy - hr_dy).abs()
    else:
        sr_mag = torch.sqrt(sr_dx*sr_dx + sr_dy*sr_dy + 1e-12)
        hr_mag = torch.sqrt(hr_dx*hr_dx + hr_dy*hr_dy + 1e-12)
        diff = (sr_mag - hr_mag).abs()

    return (diff * diff).mean() if reduction == "l2" else diff.mean()

# ────────────────────────────────────────────────────────────────
# 5. TRAINING
# ────────────────────────────────────────────────────────────────
print("Starting training…")

iter_idx = 0
plotter = TrainPlotter(save_root, ema_beta=0.9, smooth_window=200)

for ep in range(0, num_epochs + 1): #CHANGE BACK FOR FULL TRAINING
    G.train()

    for cur, hr in train_ld:
        iter_idx += 1

        patch_size = random.choice(PATCH_SET)
        cur, hr = random_patch_from_hr(hr, lr=cur, scale=4, patch_size=patch_size)
        cur, hr = [t.to(device).bfloat16() for t in (cur, hr)]

        G.requires_grad_(True)
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            sr_res = G(cur)
            bil = F.interpolate(cur, scale_factor=4, mode="bilinear", align_corners=False)

            # losses
            # Apply Yw channel-wise across spatial dimensions
            Yw = Yw_like(cur) # Shape (1, 3, 1, 1)
            sr_combined = sr_res + bil # Shape (B, 3, H, W)
            y_pred = (sr_combined * Yw).sum(1, keepdim=True) # Shape (B, 1, H, W)

            y_true = (hr * Yw).sum(1, keepdim=True) # Shape (B, 1, H, W)
            pix_y_loss = F.l1_loss(y_pred, y_true)   # optional to log later
            pix_loss   = F.l1_loss(sr_combined, hr)

            sr_rgb = yuv_to_rgb((sr_combined).clamp(0,1))
            hr_rgb = yuv_to_rgb(hr)

            # Check if w_perc is defined before using it
            f_sr = vgg(_vgg_norm(sr_rgb.float()))
            with torch.no_grad(): f_hr = vgg(_vgg_norm(hr_rgb.float()))
            loss_perc = F.l1_loss(f_sr, f_hr)

            grad_loss = sobel_loss(sr_rgb, hr_rgb, mode="mag", reduction="l1")
            res_loss  = F.l1_loss(sr_res, hr - bil)

            # weights schedule
            # if ep < 8 and not skip_warmup:
            #     w_pix, w_perc, w_res, w_grad = 1.0, 1.0, 0.25, 0.08
            # else:
            #     w_pix, w_perc, w_res, w_grad = 0.7, 0.6, 1.0, 0.04
            # t goes 0→1 over warmup_len epochs
            warmup_len = 8
            t = min(1.0, ep / float(warmup_len))
            s = ease(t)
            w_pix  = (1.75 * (1 - s)) + (1.0 * s)
            w_perc = (1.0 * (1 - s)) + (0.6 * s)
            w_res  = (0.25 * (1 - s)) + (.5 * s)
            w_grad = (0.08 * (1 - s)) + (0.04 * s)

            loss_G = (w_pix  * pix_loss
                      + w_perc * loss_perc
                      + w_res  * res_loss
                      + w_grad * grad_loss)

        opt_G.zero_grad(set_to_none=True)
        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        # logging → plots
        plotter.update_iter(
            iter_idx,
            float(loss_G.item()),
            float(loss_perc.item()),
            float(res_loss.item()),
            float(pix_loss.item()),
            float(grad_loss.item()),
            float(w_perc * loss_perc.item()),
            float(w_res  * res_loss.item()),
            float(w_pix  * pix_loss.item()),
            float(w_grad * grad_loss.item()),
        )

        if (iter_idx % 100) == 0:
            plotter.save_iter_plots()
            plotter.save_individual_iter_plots()
            plotter.save_iter_components_dualaxis()

    # end epoch
    print(f"Epoch {ep}/{num_epochs}")
    torch.save(G.state_dict(), os.path.join(save_root, f"epoch_{ep:04d}_final.G.pt"))
    print(f"Epoch {ep} finished — saved generator @ {save_root}")

    plotter.update_epoch(ep)
    plotter.save_epoch_plots()
    plotter.save_individual_epoch_plots()

    if (ep % 20 == 0) or (ep == num_epochs) or (ep == 1):
        for _ in range(2):
            demo(G, demo_loader, crp_sz=256)
            demo(G, demo_loader, crp_sz=512)
            demo(G, demo_loader, crp_sz=127)

Starting training…


RuntimeError: Given groups=1, weight of size [256, 3, 5, 5], expected input[8, 256, 32, 32] to have 3 channels, but got 256 channels instead

In [ ]:
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores